# Task API walkthrough

This notebook uses deterministic synthetic events to demonstrate aligned event selections, vertex/direction/energy/classification quantities, dependency evaluation, and plots without requiring private data files. Run it from the repository root. For a real HDF5/ML/fiTQun workflow, continue with `compare_vertex_samples.ipynb`.

## 1. Load the reusable example functions

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

REPOSITORY_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "examples" / "simple" / "task_api_walkthrough.py").is_file()
    ),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of the repository.")
os.chdir(REPOSITORY_ROOT)
script_path = REPOSITORY_ROOT / "examples" / "simple" / "task_api_walkthrough.py"
spec = importlib.util.spec_from_file_location("task_api_walkthrough", script_path)
walkthrough = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = walkthrough
spec.loader.exec_module(walkthrough)

## 2. Build quantities and apply one shared selection

The example creates event-level quantities first, then applies the same Boolean mask to every task quantity and its Wall dependency. The cut flow reports each selection stage, avoiding accidental misalignment between task arrays.

In [ ]:
data = walkthrough.build_demo_data()
data.selection.print_cut_flow()
print(f"\nSelected events: {data.vertex.event_count:,}")

## 3. Inspect task-specific quantities

Direction quantities use degrees by default; `opening_angles` is a non-negative angular separation and `phi_residuals` uses the shortest signed difference. Energy quantities expose absolute and fractional residuals. Classification scores are converted to probabilities with softmax by default, while the provided class labels map score columns to physical labels.

In [ ]:
import numpy as np

print(f"Median 3D vertex error: {np.median(data.vertex.errors_3d):.2f} cm")
print(f"Median opening angle: {np.median(data.direction.opening_angles):.2f} degree")
median_energy_error = np.median(np.abs(data.energy.residuals))
print(f"Median absolute energy residual: {median_energy_error:.2f} MeV")
print(f"Classification accuracy: {np.mean(data.classification.correct_mask):.3f}")
print(f"First event probabilities: {data.classification.probabilities[0]}")

## 4. Evaluate dependencies and visualize results

The profile example uses a 68th-percentile 3D error as resolution and reports classification accuracy with a standard error. These are deliberately explicit metric choices; choose definitions and minimum bin counts appropriate to the analysis.

In [ ]:
figures = walkthrough.make_demo_plots(data)
for name, figure in figures.items():
    print(name)
    display(figure)
    figure.tight_layout()

## Next: use your own files

For predictions stored as NumPy arrays, start with `examples/simple/load_prediction_data.py`. For the complete source-HDF5 plus ML-run selection, multiple-file comparison, optional event matching, fiTQun inputs, normalization, evaluation, and plot workflow, see `examples/full/compare_vertex_samples.py` or `compare_vertex_samples.ipynb`. Edit the paths, HDF5 dataset keys, units, and selection settings before running those data-dependent examples.